# Kompresibilna sapnica: predvidi → izračunaj → provjeri

Za konvergentnu sapnicu s idealnim plinom snižavanje protutlaka najprije povećava maseni protok. Kada se u grlu dosegne \(M=1\), daljnje snižavanje protutlaka ne povećava protok kroz ovaj model.

## Predvidi

1. Skiciraj \(\dot m(p_b/p_0)\) i označi gdje očekuješ plato.
2. Ako se površina grla udvostruči, što se događa s prigušenim protokom?
3. Hoće li povećanje stagnacijske temperature povećati ili smanjiti \(\dot m_{max}\)?

Pretpostavke su kvazijednodimenzijski, izentropski idealni tok do grla i poznat koeficijent istjecanja. Model ne opisuje udarne valove ni tok nizvodno od grla.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "font.size": 10})
gamma, R = 1.4, 287.0

def critical_pressure_ratio(gamma=gamma):
    return (2/(gamma+1))**(gamma/(gamma-1))

def choked_mass_flow(p0, T0, area, Cd=1.0, gamma=gamma, R=R):
    factor = np.sqrt(gamma/R)*(2/(gamma+1))**((gamma+1)/(2*(gamma-1)))
    return Cd*area*p0/np.sqrt(T0)*factor

def mass_flow(back_pressure_ratio, p0, T0, area, Cd=1.0):
    ratio = np.asarray(back_pressure_ratio, dtype=float)
    if np.any((ratio <= 0) | (ratio > 1)):
        raise ValueError("Treba vrijediti 0 < pb/p0 ≤ 1.")
    rcrit = critical_pressure_ratio()
    effective_ratio = np.maximum(ratio, rcrit)
    mach = np.sqrt(2/(gamma-1)*(effective_ratio**(-(gamma-1)/gamma)-1))
    flow_parameter = mach*(1+(gamma-1)*mach**2/2)**(-(gamma+1)/(2*(gamma-1)))
    dimensional = Cd*area*p0/np.sqrt(T0)*np.sqrt(gamma/R)
    return dimensional*flow_parameter

base = dict(p0=600e3, T0=300.0, area=50e-6, Cd=0.97)
rcrit = critical_pressure_ratio()
mdot_star = choked_mass_flow(**base)
print(f"Kritični omjer pb/p0 = {rcrit:.6f}")
print(f"Prigušeni maseni protok = {mdot_star:.6f} kg/s")


## Izračunaj: karakteristika i nesigurnost prigušenog protoka

Za male, neovisne ulazne nesigurnosti vrijedi približna relativna bilanca

\[
\left(\frac{u_{\dot m}}{\dot m}\right)^2=
\left(\frac{u_{p_0}}{p_0}\right)^2+
\left(\frac{u_A}{A}\right)^2+
\left(\frac{u_{C_d}}{C_d}\right)^2+
\frac14\left(\frac{u_{T_0}}{T_0}\right)^2.
\]

Provjeravamo je determinističkim Monte Carlo uzorkovanjem.


In [ ]:
ratios = np.linspace(0.08, 1.0, 300)
mdot_curve = mass_flow(ratios, **base)

sigma = dict(p0=3e3, T0=1.5, area=0.30e-6, Cd=0.005)
relative_linear = np.sqrt(
    (sigma["p0"]/base["p0"])**2 +
    (sigma["area"]/base["area"])**2 +
    (sigma["Cd"]/base["Cd"])**2 +
    0.25*(sigma["T0"]/base["T0"])**2
)
u_linear = mdot_star*relative_linear

rng = np.random.default_rng(20260802)
n_samples = 40_000
mc = {key: rng.normal(base[key], sigma[key], n_samples) for key in base}
mdot_mc = choked_mass_flow(**mc)
u_mc = np.std(mdot_mc, ddof=1)
interval = np.quantile(mdot_mc, [0.025, 0.975])
print(f"u_linear = {u_linear:.6f} kg/s; u_MC = {u_mc:.6f} kg/s")
print(f"95 %-tni Monte Carlo interval = [{interval[0]:.6f}, {interval[1]:.6f}] kg/s")


## Provjeri

Plato se provjerava na više protutlakova, geometrijsko skaliranje dvostrukom površinom, a linearna propagacija neovisnim Monte Carlo postupkom.


In [ ]:
plateau = mass_flow(np.array([0.10, 0.25, 0.50]), **base)
double_area = choked_mass_flow(**{**base, "area": 2*base["area"]})
near_one = float(mass_flow(np.array([1.0]), **base)[0])

assert np.allclose(plateau, mdot_star, rtol=1e-12)
assert np.isclose(double_area, 2*mdot_star, rtol=1e-13)
assert np.isclose(near_one, 0.0, atol=1e-14)
assert abs(u_mc/u_linear-1) < 0.05

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(ratios, mdot_curve, color="#256d85", lw=2)
axes[0].axvline(rcrit, color="#b43c35", ls="--", label="M=1 u grlu")
axes[0].set(xlabel="$p_b/p_0$", ylabel=r"$\dot m$ (kg/s)", title="Prigušenje masenog protoka")
axes[0].legend()
axes[1].hist(mdot_mc, bins=55, color="#7cb5d6", edgecolor="white")
axes[1].axvline(mdot_star, color="#b43c35", lw=2, label="nominalno")
axes[1].set(xlabel=r"$\dot m_{max}$ (kg/s)", ylabel="broj uzoraka", title="Ulazna nesigurnost")
axes[1].legend()
for ax in axes: ax.grid(True, ls=":", alpha=.45)
plt.tight_layout(); plt.show()


## Protumači

Plato nije numeričko zasićenje nego promjena fizikalnog ograničenja. Za validaciju koeficijenta istjecanja treba mjeriti stvarni maseni protok i stagnacijske veličine te iskazati njihovu nesigurnost; slaganje jedne radne točke nije dokaz valjanosti u cijelom rasponu.
